In [ ]:
from aare.constants import TIME
from aare.pd_utils import fill_with_hard_limit
import plotly.express as px
from aare_influx.remote_existenz_store import RemoteExistenzStore
from aare_train.features.air_temp import AirTemp
from aare_train.features.water_temp import WaterTemp
from aare_train.params import read_params
from aare_train.paths import DATA_FOLDER, PROJECT_ROOT

import subprocess

import polars as pl
import polars.selectors as cs
import psycopg_pool

from aare_timescale.postgres import copy_to_df_pl

# Evaluation of prod forecasts, comparison of fixed 1.1 vs old 1.0

In notebooks 22, 23 and 24, I already did various evaluations and simulations to try and fix the misalignment
I only recently thought about. I also fixed the bug of a too low diff_threshold.

Now I want to evaluate how much better the forecasts of the new model with fixed diff_threshold and fixed hour alignment
(aka using FIRST as target instead of MEAN) are compared to the forecasts of the previous nowcasting_temp 1.0 model.
It should use all the data we gathered so far and be clean in the sense that it isn't a completely unreproducible mess like the other three notebooks.

Make sure you are synced with the prod database for this.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("dulwich").setLevel(logging.WARNING)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
store = RemoteExistenzStore()

In [ ]:
pool = psycopg_pool.AsyncConnectionPool("host=127.0.0.1 dbname=aare_oraku user=postgres password=password", open=False)
await pool.open()

In [ ]:
params = read_params()
tz = params["general"]["timezone"]
data_dir = DATA_FOLDER / "isolated_tasks" / "2025-05_bug-fix-eval"
data_dir.mkdir(parents=True, exist_ok=True)
live_forecast_meta_path = data_dir / "live_forecast_meta_nowcasting_temp-1.0.parquet"
live_forecast_path = data_dir / "live_forecast_nowcasting_temp-1.0.parquet"
simulated_forecast_10_path = data_dir / "sim_forecast_nowcasting_temp-1.0.parquet"
simulated_forecast_11_path = data_dir / "sim_forecast_nowcasting_temp-1.1.parquet"
actual_raw_path = data_dir / "actual_raw.parquet"

In [ ]:
run_ts_hour = pl.col("run_ts").dt.truncate("1h").alias("run_ts_hour")
accurate_run_filter = pl.col("run_ts").dt.minute() == 15

In [ ]:
model_name = "nowcasting_temp"
model_version = "1.0"
successful_run_where = (
    f"meta.model_name = '{model_name}' and meta.model_version = '{model_version}' and status = 'success'"
)

In [ ]:
# assert not live_forecast_meta_path.exists(), f"{live_forecast_meta_path} exists already, you sure you want to run this?"
if live_forecast_meta_path.exists():
    live_forecast_meta = pl.read_parquet(live_forecast_meta_path)
else:
    async with pool.connection() as _conn:
        # noinspection PyTypeChecker
        live_forecast_meta = await copy_to_df_pl(
            _conn,
            f"""select * from forecast_meta as meta
    where {successful_run_where}
    order by run_ts
    """,
        )

        live_forecast_meta = live_forecast_meta.with_columns(pl.col("run_ts").str.to_datetime(time_zone=tz)).set_sorted(
            "run_ts"
        )

    live_forecast_meta.write_parquet(live_forecast_meta_path)

live_forecast_meta

In [ ]:
# assert not live_forecast_path.exists(), f"{live_forecast_path} exists already, you sure you want to run this?"
if live_forecast_path.exists():
    live_forecasts = pl.read_parquet(live_forecast_path)
else:
    async with pool.connection() as _conn:
        # noinspection PyTypeChecker
        live_forecasts = await copy_to_df_pl(
            _conn,
            f"""select forecast.* from forecast
    join forecast_meta as meta on forecast.run_ts = meta.run_ts
    where {successful_run_where}
    order by run_ts, time
    """,
        )

        live_forecasts = live_forecasts.with_columns(cs.string().str.to_datetime(time_zone=tz)).set_sorted(
            ["run_ts", "time"]
        )

    live_forecasts.write_parquet(live_forecast_path)

live_forecasts

In [ ]:
run_ts_list = live_forecast_meta.select(pl.col("run_ts").dt.to_string()).to_series().to_list()
run_ts_list

In [ ]:
proc_simulate_10 = None
proc_simulate_11 = None

In [ ]:
assert not simulated_forecast_11_path.exists(), "simulation file already exists"
assert proc_simulate_11 is None, "simulation already started! cleanup first."
assert WaterTemp("BERN").field.agg_fn == "first" and AirTemp("BERN").field.agg_fn == "first", (
    "WON'T START SIMULATION of nowcasting_temp-1.1, if the features aren't configured to use 'first' aggregation. Temporarily change the features before running this!"
)

proc_simulate_11 = subprocess.Popen(
    "uv run src/oraku-forecast/main.py --logging-level INFO --model-path data/models/nowcasting_temp-1.1/nowcasting_temp-1.1.json ".split()
    + ["--simulation-file", str(simulated_forecast_11_path.absolute())]
    + ["--simulate-runts"]
    + run_ts_list,
    cwd=PROJECT_ROOT,
)

In [ ]:
if proc_simulate_11.poll() is not None:
    print("Simulation for nowcasting_temp-1.1 has finished with exit code:", proc_simulate_11.returncode)
else:
    print("Simulation for nowcasting_temp-1.1 is still running")

In [ ]:
assert proc_simulate_11 is None or proc_simulate_11.poll() is not None, "won't cleanup a running process"
proc_simulate_11 = None

In [ ]:
sim_forecasts_11 = pl.read_parquet(simulated_forecast_11_path)
sim_forecasts_11

In [ ]:
# same for live model (nowcasting_temp-1.0)

In [ ]:
assert not simulated_forecast_10_path.exists(), "simulation file already exists"
assert proc_simulate_10 is None, "simulation already started! cleanup first."
assert WaterTemp("BERN").field.agg_fn == "mean" and AirTemp("BERN").field.agg_fn == "mean", (
    "WON'T START SIMULATION of nowcasting_temp-1.0, if the features aren't configured to use 'mean' aggregation. Temporarily change the features before running this!"
)

proc_simulate_10 = subprocess.Popen(
    "uv run src/oraku-forecast/main.py --logging-level INFO --model-path data/models/nowcasting_temp-1.0/nowcasting_temp-1.0.json ".split()
    + ["--simulation-file", str(simulated_forecast_10_path.absolute())]
    + ["--simulate-runts"]
    + run_ts_list,
    cwd=PROJECT_ROOT,
)

In [ ]:
if proc_simulate_10.poll() is not None:
    print("Simulation for nowcasting_temp-1.0 has finished with exit code:", proc_simulate_10.returncode)
else:
    print("Simulation for nowcasting_temp-1.0 is still running")

In [ ]:
assert proc_simulate_10 is None or proc_simulate_10.poll() is not None, "won't cleanup a running process"
proc_simulate_10 = None

In [ ]:
sim_forecasts_10 = pl.read_parquet(simulated_forecast_10_path)
sim_forecasts_10

In [ ]:
start_time, end_time = (
    live_forecasts.select(pl.min("time").alias("min"), pl.max("time").alias("max")).to_dicts()[0].values()
)
start_time, end_time

In [ ]:
if actual_raw_path.exists():
    actual_raw = pl.read_parquet(actual_raw_path)
else:
    actual_raw_df = store.query((start_time, end_time), "hydro/temperature:raw_?@bern")
    # see interpolation check below. up to 2 hours can easily be interpolated linearly.
    # the configuration of the temp_bern feature interpolates even larger gaps anyway (on hourly data)
    upsampled = actual_raw_df.resample("10min", on=TIME, closed="left", label="left").first()
    interpolated = fill_with_hard_limit(upsampled, 2 * 6, columns=["temperature_bern"], method="linear")

    actual_raw = pl.from_dataframe(interpolated)
    actual_raw.write_parquet(actual_raw_path)

actual_raw

In [ ]:
# should be evenly distributed now thanks to upsampling
actual_raw.select(pl.col(TIME).dt.minute().value_counts())

In [ ]:
df = store.query((start_time, end_time), "hydro/temperature:raw_?@bern")
df

In [ ]:
upsampled = df.resample("10min", on=TIME, closed="left", label="left").first().reset_index()
upsampled

In [ ]:
upsampled.isna().sum()

In [ ]:
interpolated = fill_with_hard_limit(upsampled, 1 * 6, columns=["temperature_bern"], method="linear")
interpolated

In [ ]:
interpolated.isna().sum()

In [ ]:
px.scatter(upsampled.join(interpolated, rsuffix="_filled"), x=TIME, y=["temperature_bern_filled", "temperature_bern"])

## Simulation with ingestion delay

Due to time pressure, I didn't implement this cleanly and instead just used the following patch on top of d7e226a to temporarily enable simulation
with correct ingestion delay. Unfortunately, I do need it to compare timing in notebook 26.

```
diff --git a/src/oraku-forecast/src/oraku_forecast/data/compile_data.py b/src/oraku-forecast/src/oraku_forecast/data/compile_data.py
index d3eee7d..2238202 100644
--- a/src/oraku-forecast/src/oraku_forecast/data/compile_data.py
+++ b/src/oraku-forecast/src/oraku_forecast/data/compile_data.py
@@ -162,7 +162,18 @@ def _pull_influx(
     # take data from a bit further in the past than theoretically necessary to definitely get all the required data,
     # darts handles truncation at start and end, so this is just to be safe and shouldn't cause any issues.
     hours_back = abs(min_lag) + extra_past_hours
-    period = run_ts - timedelta(hours=hours_back), run_ts  # (start, end)
+
+    ingestion_delays = {
+        "hydro": timedelta(minutes=8),
+        "smn": timedelta(minutes=15),
+    }
+    end = run_ts
+    for meas, delay in ingestion_delays.items():
+        if all(field.measurement == meas for field in fields):
+            end = end - delay
+            # print(f"[HACKY] all measurements are '{meas}', settings fetch end back by '{delay}'")
+
+    period = run_ts - timedelta(hours=hours_back), end  # (start, end)

     return influx_store.query(period, fields)

```

In [ ]:
proc_simulate_11_ing = None

In [ ]:
assert (
    "ingestion_delays =" in (PROJECT_ROOT / "src/oraku-forecast/src/oraku_forecast/data/compile_data.py").read_text()
), "MAKE SURE THE INGESTION DELAY PATCH IS APPLIED!"
output_file = simulated_forecast_11_path.with_stem(simulated_forecast_11_path.stem + "_ingestion-delay")
assert not output_file.exists(), "simulation file already exists"
assert proc_simulate_11_ing is None, "simulation already started! cleanup first."
assert WaterTemp("BERN").field.agg_fn == "first" and AirTemp("BERN").field.agg_fn == "first", (
    "WON'T START SIMULATION of nowcasting_temp-1.1, if the features aren't configured to use 'first' aggregation. Temporarily change the features before running this!"
)

proc_simulate_11_ing = subprocess.Popen(
    "uv run src/oraku-forecast/main.py --logging-level INFO --model-path data/models/nowcasting_temp-1.1/nowcasting_temp-1.1.json ".split()
    + ["--simulation-file", str(output_file.absolute())]
    + ["--simulate-runts"]
    + run_ts_list,
    cwd=PROJECT_ROOT,
)

In [ ]:
proc_simulate_11_ing